In [1]:
import json
import random
from openai import OpenAI
import os
import re
import numpy as np
import base64
import time
import pandas as pd 
from tqdm import tqdm
from word2number import w2n
from llamaapi import LlamaAPI

# Load dataset

In [2]:
def load_dataset(annotation_path, question_path):
    try:
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)['annotations']

        with open(question_path, 'r') as f:
            questions = json.load(f)['questions']

        # Select the dataset where ‘overall_scores’ == 1.0
        filtered_annotations = [
            annotation for annotation in annotations
            if annotation.get('overall_scores', {}).get('question') == 1.0 and
               annotation.get('overall_scores', {}).get('answer') == 1.0
        ]

        # Create a mapping from question ID to answer
        question_id_to_answer = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation['answer_type']  
            }
            for annotation in filtered_annotations
        }

        # Create a mapping from question ID to answer type
        question_id_to_answer_type = {
            annotation['id']: {
                'answer': annotation['answer'],
                'answer_type': annotation.get('answer_type', 'other')
            }
            for annotation in filtered_annotations
        }

        filtered_questions = [
            question for question in questions 
            if question['id'] in question_id_to_answer
        ]

        return filtered_questions, filtered_annotations, question_id_to_answer, question_id_to_answer_type

    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], [], {}, {}

def get_dataset(questions, question_id_to_answer, fraction=0.001, seed=42):
    # TODO：Increase quantity
# def get_dataset(questions, question_id_to_answer, fraction=0.05, seed=42):
    try:
        random.seed(seed)
        sample_size = max(1, int(len(questions) * fraction))
        sampled_questions = random.sample(questions, sample_size)
        sampled_truth_answers = [
            question_id_to_answer[q['id']]['answer'] 
            for q in sampled_questions
        ]
        
        for q in sampled_questions:
            q['answer_type'] = question_id_to_answer[q['id']]['answer_type']

        return sampled_questions, sampled_truth_answers

    except Exception as e:
        print(f"Error sampling dataset: {e}")
        return [], []

# Use first dataset
# def get_dataset(questions, annotations, fraction=0.05):
#     sample_size = int(len(questions) * fraction)
#     sampled_questions = questions[:sample_size]
#     sampled_annotations = annotations[:sample_size]
#     return sampled_questions, sampled_annotations

def encode_image(image_path):
    try:
        if not os.path.exists(image_path):
            print(f"Error: The image file at {image_path} was not found.")
            return None

        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    except Exception as e:
        print(f"An error occurred while encoding the image: {e}")
        return None

def parse_answer(input_str):
    if input_str is None:
        return None

    try:
        input_str = str(input_str).lower().strip()
        words = input_str.split()
        for i in range(len(words)):
            for j in range(i + 1, len(words) + 1):
                substring = ' '.join(words[i:j])
                try:
                    return str(w2n.word_to_num(substring))
                except:
                    continue

        matches = re.findall(r'\d+', input_str)
        if matches:
            return matches[-1]

        if "yes" in input_str:
            return "yes"
        elif "no" in input_str:
            return "no"

        return input_str

    except Exception as e:
        print(f"Error parsing answer '{input_str}': {e}")
        return input_str

# Single agent prediction

In [ ]:
client = OpenAI(
    api_key="",
    base_url="https://api.llama-api.com"  
)

def get_predict(question, image_base64, max_retries=3, retry_delay=2):
    if image_base64 is None:
        return None
        
    prompt = f"Question: {question}\nProvide an answer based on the image:"
    
    for attempt in range(max_retries):
        try:
            # 调试输出API请求结构（Base64仅输出部分内容）
            print("Sending API request with structure:")
            print({
                "model": "llama3.2-90b-vision",
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": question},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64[:50]}..."}}
                        ]
                    }
                ]
            })

            completion = client.chat.completions.create(
                model="llama3.2-90b-vision",
                messages=[
                    {
                        "role": "user",
                        "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
                        ],
                    }
                ],
                max_tokens=100,
                temperature=0.3,
                stream=False,
            )
            
            return completion.choices[0].message.content.strip()
            
        except Exception as e:
            print(f"尝试 {attempt+1} 失败: {str(e)}")
            time.sleep(retry_delay)
            continue
            
    print(f"所有 {max_retries} 次尝试都失败，问题: {question}")
    return None

# client = LlamaAPI("")

# def get_predict(question, image_base64, max_retries=3, retry_delay=2):
#     if image_base64 is None:
#         return None
        
#     prompt = f"Question: {question}\nProvide an answer based on the image:"
    
#     for attempt in range(max_retries):
#         try:
#             # 使用Llama的原生API格式
#             api_request_json = {
#                 "model": "llama3.2-90b-vision",
#                 "messages": [{
#                     "role": "user", 
#                     "content": [
#                         {"type": "text", "text": prompt},
#                         {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}}
#                     ]
#                 }],
#                 "max_tokens": 100,
#                 "temperature": 0.3
#             }
            
#             response = client.run(api_request_json)
#             response_json = response.json()
            
#             # 调试输出，查看响应结构
#             print(f"API响应结构: {json.dumps(response_json, indent=2)[:200]}...")
            
#             # 根据实际响应结构访问内容
#             if "error" in response_json:
#                 print(f"API错误: {response_json['error']}")
#                 time.sleep(retry_delay)
#                 continue
                
#             # 尝试不同的访问方式
#             if "choices" in response_json and len(response_json["choices"]) > 0:
#                 # 检查choices是列表还是字典
#                 choices = response_json["choices"]
#                 if isinstance(choices, list):
#                     if "message" in choices[0]:
#                         return choices[0]["message"]["content"]
#                     elif "text" in choices[0]:
#                         return choices[0]["text"]
#                 elif isinstance(choices, dict):
#                     if "message" in choices:
#                         return choices["message"]["content"]
#                     elif "text" in choices:
#                         return choices["text"]
            
#             # 如果找不到预期结构，尝试查找其他可能的内容字段
#             if "content" in response_json:
#                 return response_json["content"]
            
#             # 完全找不到内容字段，返回整个JSON以供调试
#             print("找不到内容字段，返回完整响应进行检查")
#             return str(response_json)
            
#         except Exception as e:
#             print(f"尝试 {attempt+1} 失败: {str(e)}")
#             if attempt == 0:  # 仅在第一次尝试失败时打印完整错误
#                 import traceback
#                 print(f"完整错误: {traceback.format_exc()}")
#             time.sleep(retry_delay)
#             continue
            
#     print(f"All {max_retries} attempts failed for question: {question}")
#     return None

# Calculate accuracy

In [4]:
def compute_accuracy(question, truth_answer, predicted_answer, answer_type, max_retries=2, retry_delay=2):
    """Evaluate answer accuracy using a non-vision Llama model"""
    if predicted_answer is None:
        return 0
    
    # Construct evaluation prompt
    prompt = f"""
Evaluate if the following predicted answer is correct:

Question: {question}
True answer: {truth_answer}
Predicted answer: {predicted_answer}
Answer type: {answer_type}

Rules:
1. For yes/no type questions, determine if the predicted answer conveys the same meaning as the true answer.
2. For number type questions, check if the predicted answer contains the same number as the true answer.
3. For other type questions, determine if the predicted answer contains the key information from the true answer.
4. The predicted answer may be more detailed, but it's correct if it contains the right information.

Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
- 1.0: Perfect match or completely correct meaning
- 0.75: Mostly correct with minor differences
- 0.5: Partially correct
- 0.25: Slightly correct but missing key points
- 0.0: Completely incorrect or unrelated

Return only the numeric score, no explanation.
"""
    
    for attempt in range(max_retries):
        try:
            # Use a text-only model for evaluation since we don't need vision capabilities here
            # This might be more reliable
            completion = client.chat.completions.create(
                model="llama3.2-90b-vision",  
                messages=[{"role": "user", "content": prompt}],
                max_tokens=10,
                temperature=0.3
            )
            
            response_text = completion.choices[0].message.content.strip()
            
            # Try to extract a float value
            try:
                # First, try direct conversion
                return float(response_text)
            except ValueError:
                # If that fails, try to extract numeric parts
                import re
                numbers = re.findall(r"[0-9.]+", response_text)
                if numbers:
                    return float(numbers[0])
                else:
                    print(f"Could not extract a numeric score from: '{response_text}'")
                    # Default scores based on text
                    if "perfect" in response_text.lower() or "completely" in response_text.lower():
                        return 1.0
                    elif "mostly" in response_text.lower():
                        return 0.75
                    elif "partial" in response_text.lower():
                        return 0.5
                    elif "slightly" in response_text.lower():
                        return 0.25
                    else:
                        return 0.0
                
        except Exception as e:
            print(f"Accuracy calculation attempt {attempt+1} failed: {str(e)}")
            time.sleep(retry_delay)
            continue
    
    return 0.0

# def compute_accuracy(question, truth_answer, predicted_answer, answer_type, max_retries=2, retry_delay=2):
#     """Use LlamaAPI to evaluate the accuracy of the answer"""
#     if predicted_answer is None:
#         return 0
    
#     # Construct evaluation prompt
#     prompt = f"""
# Evaluate if the following predicted answer is correct:

# Question: {question}
# True answer: {truth_answer}
# Predicted answer: {predicted_answer}
# Answer type: {answer_type}

# Rules:
# 1. For yes/no type questions, determine if the predicted answer conveys the same meaning as the true answer.
# 2. For number type questions, check if the predicted answer contains the same number as the true answer.
# 3. For other type questions, determine if the predicted answer contains the key information from the true answer.
# 4. The predicted answer may be more detailed, but it's correct if it contains the right information.

# Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
# - 1.0: Perfect match or completely correct meaning
# - 0.75: Mostly correct with minor differences
# - 0.5: Partially correct
# - 0.25: Slightly correct but missing key points
# - 0.0: Completely incorrect or unrelated

# Return only the numeric score, no explanation.
# """
    
#     for attempt in range(max_retries):
#         try:
#             # 使用LlamaAPI的原生格式
#             api_request_json = {
#                 "model": "llama3.2-90b",  # 使用非视觉模型进行评估
#                 "messages": [{
#                     "role": "user", 
#                     "content": prompt
#                 }],
#                 "max_tokens": 10,
#                 "temperature": 0.3
#             }
            
#             response = client.run(api_request_json)
#             response_json = response.json()
            
#             # 调试输出
#             print(f"计算精度API响应: {json.dumps(response_json, indent=2)[:200]}...")
            
#             # 根据实际响应结构访问内容
#             if "error" in response_json:
#                 print(f"API错误: {response_json['error']}")
#                 time.sleep(retry_delay)
#                 continue
            
#             # 尝试多种可能的响应结构
#             content = None
#             if "choices" in response_json and len(response_json["choices"]) > 0:
#                 choices = response_json["choices"]
#                 if isinstance(choices, list):
#                     if "message" in choices[0]:
#                         content = choices[0]["message"]["content"]
#                     elif "text" in choices[0]:
#                         content = choices[0]["text"]
#                 elif isinstance(choices, dict):
#                     if "message" in choices:
#                         content = choices["message"]["content"]
#                     elif "text" in choices:
#                         content = choices["text"]
            
#             if content:
#                 # 尝试从内容中提取数字
#                 try:
#                     # 移除所有非数字字符
#                     numeric_str = ''.join(c for c in content if c.isdigit() or c == '.')
#                     if numeric_str:
#                         return float(numeric_str)
#                     else:
#                         print(f"无法从响应中提取数字: {content}")
#                         return 0.0
#                 except ValueError:
#                     print(f"无法将响应转换为浮点数: {content}")
#                     return 0.0
            
#             return 0.0  # 默认返回零分
                
#         except Exception as e:
#             print(f"精度计算尝试 {attempt+1} 失败: {str(e)}")
#             time.sleep(retry_delay)
#             continue
    
#     return 0.0  # 如果所有尝试都失败，返回零分

# Load paths
annotation_path = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/simpsons/v1_Annotation_Val_simpsons_vqa.json"
question_path = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/simpsons/v1_Question_Val_simpsons_vqa.json"
images_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/simpsons/val_images"

try:
    # Load dataset
    questions, annotations, question_id_to_answer, question_id_to_answer_type = load_dataset(annotation_path, question_path)

    if not questions:
        print("The 'questions' list is empty or not a list.")
        raise ValueError("Questions list is empty")

    # Get sample data TODO
    # sampled_questions, sampled_truth_answers = get_dataset(questions, question_id_to_answer, fraction=0.05)
    sampled_questions, sampled_truth_answers = get_dataset(questions, question_id_to_answer, fraction=0.001)

    if not sampled_questions:
        print("Failed to sample questions or empty sample")
        raise ValueError("No sampled questions")

    # Initialize results storage
    accuracies = []
    evaluation_results = []
    # Create an empty collection to store the processed problem IDs
    processed_question_ids = set() 

    # Process each question
    for question, truth_answer in tqdm(zip(sampled_questions, sampled_truth_answers),
                                     total=len(sampled_questions)):
        try:
            question_id = question['id']
            # Skip if already processed this question
            if question_id in processed_question_ids:
                continue
                
            processed_question_ids.add(question_id)
            
            question_text = question['question']
            image_relative_path = question['img_path']
            answer_type = question_id_to_answer_type[question_id]['answer_type']

            image_path = os.path.join(images_dir, image_relative_path)
            image_base64 = encode_image(image_path)

            if image_base64 is None:
                print(f"Skipping question ID {question_id} due to image encoding failure")
                continue

            model_answer = get_predict(question_text, image_base64)
            pred_solutions = [model_answer] if model_answer is not None else []

            if not pred_solutions:
                print(f"No prediction obtained for question ID {question_id}")
                continue

            accuracy = compute_accuracy(
                question=question_text,
                truth_answer=truth_answer,
                predicted_answer=model_answer,
                answer_type=answer_type
            )

            # Print results
            print(f"Question ID: {question_id}")
            print(f"Question: {question_text}")
            print(f"Answer Type: {answer_type}") 
            print(f"Truth Answer: {truth_answer}")
            if model_answer is not None:
                print(f"Predicted Answer: {model_answer}")
            if accuracy is not None:
                accuracies.append(accuracy)
                print(f"Accuracy: {accuracy:.4f}")
            else:
                print(f"Warning: No accuracy for question: {question_text}")

            # Store result
            result = {
                'question_id': question_id,
                'question': question_text,
                'answer_type': answer_type,
                'truth_answer': truth_answer,
                'predicted_answer': model_answer,
                'accuracy': accuracy
            }
            evaluation_results.append(result)
            accuracies.append(accuracy)

        except Exception as e:
            print(f"Error processing question {question.get('id', 'unknown')}: {e}")
            continue

    # Calculate average accuracy
    average_accuracy = np.mean(accuracies) if accuracies else 0
    print(f"Average Accuracy: {average_accuracy:.4f}")

except Exception as e:
    print(f"Unexpected error: {e}")
    average_accuracy = 0

  0%|          | 0/7 [00:00<?, ?it/s]

Sending API request with structure:
{'model': 'llama3.2-90b-vision', 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'what is on the shelf?'}, {'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,/9j/4ROQRXhpZgAATU0AKgAAAAgABwESAAMAAAABAAEAAAEaAA...'}}]}]}
尝试 1 失败: Error code: 422 - ['Some error occurred during the requisition.']
Sending API request with structure:
{'model': 'llama3.2-90b-vision', 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'what is on the shelf?'}, {'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,/9j/4ROQRXhpZgAATU0AKgAAAAgABwESAAMAAAABAAEAAAEaAA...'}}]}]}
尝试 2 失败: Error code: 422 - ['Some error occurred during the requisition.']
Sending API request with structure:
{'model': 'llama3.2-90b-vision', 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'what is on the shelf?'}, {'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,/9j/4ROQRXhpZgAATU0AKgAAAAgABwESAAMAAAABA

 14%|█▍        | 1/7 [00:09<00:57,  9.62s/it]

所有 3 次尝试都失败，问题: what is on the shelf?
No prediction obtained for question ID 77311
Sending API request with structure:
{'model': 'llama3.2-90b-vision', 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'how many people are in the picture?'}, {'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,/9j/4Qy1RXhpZgAATU0AKgAAAAgABwESAAMAAAABAAEAAAEaAA...'}}]}]}
尝试 1 失败: Error code: 422 - ['Some error occurred during the requisition.']
Sending API request with structure:
{'model': 'llama3.2-90b-vision', 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'how many people are in the picture?'}, {'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,/9j/4Qy1RXhpZgAATU0AKgAAAAgABwESAAMAAAABAAEAAAEaAA...'}}]}]}
尝试 2 失败: Error code: 422 - ['Some error occurred during the requisition.']


 14%|█▍        | 1/7 [00:12<01:17, 12.91s/it]


KeyboardInterrupt: 

# Save results

In [ ]:
# Save results with explicit file handling to ensure overwriting works
evaluation_results = [r for r in evaluation_results if r['question_id'] != 'Average']
# Ensures no duplicate summary rows when saving results
unique_questions = len(set(r['question_id'] for r in evaluation_results))

# Add average accuracy as the last row
average_result = {
    'question_id': 'Average',
    'question': f'Total Questions: {unique_questions}',  
    'answer_type': 'All',  
    'truth_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy 
}
evaluation_results.append(average_result)

column_order = [
    'question_id',
    'question',
    'answer_type',
    'truth_answer',
    'predicted_answer',
    'accuracy'
]

# Save to CSV
results_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/results"
os.makedirs(results_dir, exist_ok=True)
output_path = os.path.join(results_dir, f'simpsons_evaluation_results_single_agent_{model_name}.csv')

# First, check if the file exists and explicitly remove it
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Convert to DataFrame and save with error handling
try:
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df[column_order]
    
    # Save with explicit file opening to ensure it closes properly
    results_df.to_csv(output_path, index=False)
    
    # Verify the file was created
    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")